# Clustering: Choosing a scVI Latent Dimension

Loads the scVI-integrated counts, compares clustering/UMAP across a few
candidate latent dimensions, then finalizes on one choice and saves the
clustered output.

## Setup

In [ ]:
import os

os.environ["NUMBA_CACHE_DIR"] = "/path/to/tmp"
os.environ["MPLCONFIGDIR"] = "/path/to/tmp"

import scanpy as sc

sc.settings.verbosity = 1  # errors (0), warnings (1), info (2), hints (3)
sc.settings.figdir = "./figures-cellbender/integration/1_scvi/"
sc.settings.set_figure_params(dpi=80)

## Config

In [ ]:
input_path = "/path/to/output/S3_scvi_adata_merge.h5ad"
output_path = "/path/to/output/S4_scVI_integrated_adata_cluster_latent10.h5ad"

# candidate scVI latent dimensions to compare
candidate_latents = [10, 20, 30]
n_neighbors = 40
leiden_resolution = 0.3

# QC/metadata columns to color UMAPs by while comparing candidates
compare_color_keys = ["leiden_scvi", "ID", "n_genes", "scrublet_score", "group2"]

# once a latent dimension is chosen, use a broader set of columns for the final plot
final_color_keys = [
    "leiden_scvi", "sample", "batchseq", "n_genes",
    "scrublet_score", "percent_mito", "phase", "group2",
]

final_latent = 10

## Load and preprocess data

In [ ]:
adata = sc.read(input_path)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata

## Compare candidate latent dimensions

In [ ]:
def cluster_with_latent(adata, latent, n_neighbors, resolution, color_keys, save_suffix=None):
    """Recompute neighbors/UMAP/Leiden from a given scVI latent representation
    and plot the requested metadata columns."""
    latent_key = f"X_scVI_n_latent_{latent}"
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep=latent_key)
    sc.tl.umap(adata)
    sc.tl.leiden(adata, key_added="leiden_scvi", resolution=resolution)

    save_arg = f"_scVI_latent{latent}_all.pdf" if save_suffix else None
    sc.pl.umap(adata, color=color_keys, color_map="OrRd", ncols=4, wspace=0.3, save=save_arg)
    return adata

In [ ]:
for latent in candidate_latents:
    print(f"Clustering with latent dimension {latent}")
    adata = cluster_with_latent(
        adata, latent, n_neighbors, leiden_resolution, compare_color_keys, save_suffix=True
    )

## Finalize clustering with the chosen latent dimension

After comparing the candidates above, `final_latent` is selected for downstream analysis.

In [ ]:
adata = cluster_with_latent(
    adata, final_latent, n_neighbors, leiden_resolution, final_color_keys, save_suffix=True
)

In [ ]:
sc.pl.umap(
    adata, color=["leiden_scvi"], legend_loc="on data", legend_fontsize="xx-small",
    size=4, save=f"_scVI_latent{final_latent}_clustersOnData_all.pdf",
)

## Save clustered output

In [ ]:
adata.X = adata.layers["raw_counts"].copy()
adata.write(output_path)
adata

## QC: gene counts per donor/sample

In [ ]:
sc.set_figure_params(figsize=(8, 8))
sc.pl.violin(adata, ["n_genes"], groupby="donor", rotation=90)
sc.pl.violin(adata, ["n_genes"], groupby="ID", rotation=90)